In [0]:
%run ./funcoes_comuns

In [0]:
CATALOGO = "workspace"

uc = ler_bronze(CATALOGO, "unidades_consumidoras")
print(f"{uc.count():,} linhas")
uc.printSchema()

In [0]:
perfilar_coluna(uc, "cep")

In [0]:
for c in ["id_uc", "cpf_cnpj", "nome_titular"]:
    perfilar_coluna(uc, c, n=5)

In [0]:
fora_padrao = uc.filter(~F.col("id_uc").rlike(r"^UC\d{6}$") & (F.col("id_uc") != "id_uc"))

print(f"{fora_padrao.count():,} linhas com id_uc fora do padrão")
display(fora_padrao.groupBy("cpf_cnpj", "nome_titular").count())

In [0]:
for c in ["logradouro", "numero", "bairro", "municipio"]:
    perfilar_coluna(uc, c, n=5)

In [0]:
for c in ["uf", "classe", "subgrupo_tarifario", "status", "alimentador"]:
    perfilar_coluna(uc, c, n=5)

In [0]:
uc_tipos = (remover_cabecalho_repetido(uc)
    .withColumn("data_ligacao", parse_data("data_ligacao"))
    .withColumn("latitude", parse_decimal("latitude", 9, 6))
    .withColumn("longitude", parse_decimal("longitude", 9, 6)))

for c in ["data_ligacao", "latitude", "longitude"]:
    perfilar_coluna(uc_tipos, c, n=5)

In [0]:
decada = (F.floor(F.year("data_ligacao") / 10) * 10).alias("decada")
em_1_jan = (F.date_format("data_ligacao", "MM-dd") == "01-01").cast("int")

display(uc_tipos.groupBy(decada)
        .agg(F.count("*").alias("ucs"), F.sum(em_1_jan).alias("em_1_jan"))
        .orderBy("decada"))

In [0]:
md = ler_bronze(CATALOGO, "medidores")
print(f"{md.count():,} linhas")

for c in ["id_medidor", "modelo", "tipo", "status",
          "data_instalacao", "data_retirada", "data_ultima_manutencao"]:
    perfilar_coluna(md, c, n=5)

## Resultados da exploração (unidades e medidores)

### Unidades consumidoras (Bronze: 46.461 linhas)

| Achado | Quantidade | Regra na Silver |
|---|---|---|
| Cabeçalho repetido no meio do arquivo | 1 linha | remover (REMOVIDO) |
| Registros de teste: `id_uc` TESTE0000 a TESTE0199, todos com CPF `00000000000` e nome de preenchimento (`a`, `XXXX`, `TESTE`, `TESTE TESTE`); nenhum aparece em faturas, leituras, medidores ou ocorrências | 200 linhas | `id_uc` fora de `^UC\d{6}$` → QUARENTENA |
| IDs repetidos que diferem só na caixa do nome e na máscara do CPF/CNPJ | — | padronizar → deduplicar → 45.000 UCs únicas |
| CEP sem valor em 8 formas: nulo, `N/A`, `NULL`, `null`, espaços, `-`, `-1`, `9999-12-31`. Os válidos têm sempre 8 dígitos na faixa de PE (50 a 56) | 1.383 linhas | → nulo (CORRIGIDO) |
| CPF/CNPJ com máscara (11 dígitos = CPF; 14 = CNPJ) | 4.599 linhas | somente dígitos (CORRIGIDO) |
| Logradouro com espaços nas pontas e espaços duplos | ~2.800 linhas | aparar + padronizar (CORRIGIDO) |
| Bairro com encoding quebrado (UTF-8 lido como Latin-1, ex.: `Jardim JordÃ£o`); todos têm a versão correta no próprio arquivo | 59 bairros | reverter o encoding (CORRIGIDO) |
| UF em 5 grafias | — | → `PE` (CORRIGIDO) |
| Classe em 15 grafias (caixa e acento) | — | padronizar → 5 classes (CORRIGIDO) |
| Data de ligação entre 1900 e 1949, **sempre em 1º de janeiro**; nenhuma ligação entre 1950 e 1989; nunca coincide com a instalação do 1º medidor (nas demais UCs, coincide em 92%) | 90 UCs | data → nulo + `flag_data_ligacao_suspeita` (FLAG) |

**Limpos:** município (14 valores), alimentador (122, formato `ALM-XXX-999`), subgrupo (B1, B2, B3, A4), status (só `ATIVA`), número (só dígitos) e coordenadas (todas na RMR: latitude −8,46 a −7,70; longitude −35,14 a −34,78).

**Consistência (100%):** classe ↔ tipo de documento (residencial e rural com CPF; demais com CNPJ); classe ↔ subgrupo (RESIDENCIAL-B1, COMERCIAL-B3, INDUSTRIAL-A4, RURAL-B2, PODER PUBLICO-B3); prefixo do alimentador ↔ município.

**Resultado esperado:** 45.000 UCs na Silver (o número do case) e 200 na quarentena.

### Medidores (Bronze: 51.812 linhas)

| Achado | Quantidade | Regra na Silver |
|---|---|---|
| Duplicatas idênticas | 513 linhas | deduplicar → 51.299 medidores (REMOVIDO) |
| Modelo com espaços nas pontas | — | aparar (CORRIGIDO) |
| Tipo em 10 grafias | — | → ELETROMECANICO, ELETRONICO, SMART (CORRIGIDO) |
| Data de instalação em 4 formatos: `yyyy-MM-dd`, `MM-dd-yyyy` (hífen = mês-dia: nenhuma das ~1.250 tem o 1º número acima de 12), `dd/MM/yyyy` e `dd.MM.yyyy` | 3.728 linhas | `parse_data` (CORRIGIDO) |
| Data de retirada vazia ou `9999-12-31`, as duas significando "sem retirada" | 45.445 linhas = total de ATIVOS | → nulo (CORRIGIDO) |
| `data_ultima_manutencao` 100% vazia na origem | 51.812 linhas | manter a coluna nula (MANTIDO) |
| "445 UCs com mais de um medidor ativo" | — | eram as duplicatas: depois de deduplicar, nenhuma. A `flag_multiplos_ativos` foi descartada |

**Consistência (100%):** exatamente 1 medidor ativo por UC (45.000); ATIVO ⇔ sem retirada; em todas as 6.299 trocas, a retirada do antigo é no mesmo dia da instalação do novo; cada modelo pertence sempre ao mesmo tipo; 0 medidores de UC inexistente.

### Outros
- **Colunas deslocadas:** nenhuma. Todas as linhas dos CSVs têm exatamente o número de campos do cabeçalho.
- **Insight para o diagnóstico e a fila de perdas:** entre set/2025 e mar/2026 houve uma campanha de troca de medidores. Foram 6.299 trocas, 99% delas para SMART (5.661 substituíram eletromecânicos). Como medidor eletromecânico antigo tende a registrar menos que o consumo real, a troca pode afetar o volume medido em P1.